# search_race_uids.py

In [ ]:
"""
This script checks for the existence of UTMB.world race UIDs within a UID range
(default: 0 - 100_000) and time range (2014-2024). It's not meant to be run as 
a notebook.

For example:
    https://utmb.world/utmb-index/races/1..2014      -->  404 Does not exist
    https://utmb.world/utmb-index/races/10439..2023  -->  201 Does exist

Scraping all possible races:
The script stores an empty file when the request for the first year is sent and
waits for the response. If the response is 404, it will try again for the next
years until the year limit is reached. If the server does not response with a
404 status, it means that the race exists and the script continues to a next
race UID.

Parallelization:
For default settings, the lower bound number of requests is 100_000, the upper
bound is 1_000_000. With a request per second, this would take about 10 days to
check all options. However, because an empty file is stored, we can easily run
multiple instances in of this lightweight script in parallel.

Final result:
Once the script completes, all non-empty files are aggregated to an array and
stored to 'utmb-race-uids.json', the final output of this script.

MGPoirot, May 2024
"""

import requests
from tqdm import tqdm
from pathlib import Path
import json


def touch(filename):
    with open(filename, 'w') as f:
        pass


def json_out(obj: dict | list, target: str | Path):
    with open(target, 'w') as file:
        json.dump(obj, file, indent=4, sort_keys=True)


# Parameters
root = Path('D:/')
save_dir = root / 'race_details'
save_dir.mkdir(exist_ok=True)
max_race_uid = 100_000
years = 2014, 2025

# Search for all race UIDs under max_race_uid held within years
for race_uid in tqdm(range(max_race_uid)):
    race_file = save_dir / f'utmb_race_{race_uid}.json'
    if race_file.is_file():
        continue
    else:
        touch(race_file)

    for year in range(*years):
        url = f'https://utmb.world/utmb-index/races/{race_uid}..{year}'
        response = requests.get(url)
        if response.status_code == 404:
            continue
        json_out({'Status': 200}, race_file)
        break  # This assumes race_UIDs are constant over the years

# This section aggregates all race results that did not return a 404
race_uids = []
for race_uid in tqdm(range(max_race_uid)):
    file_path = save_dir / f"utmb_race_{race_uid}.json"
    # Check if the file is empty
    if file_path.stat().st_size == 0:
        continue
    else:
        race_uids.append(race_uid)
json_out(race_uids, root / 'utmb-race-uids.json')


# get_race_results.py

In [ ]:
import requests
from bs4 import BeautifulSoup
import numpy as np
from pathlib import Path
from utils import json_in, json_out
from itertools import product


def time2float(time_str: str) -> float:
    """
    PURPOSE:
    To convert time as string to time as float.
    
    ARGUMENTS:
    :param time_str: Time formatted as HH:MM:SS
    
    RETURNS: 
    :return: None
    
    EXAMPLE:
    >>> time2float('0:15:00')
    0.25
    """
    return float(np.sum([int(v) / 60 ** p for p, v in enumerate(time_str.split(':'))]))


def fmt(info: dict | False | None = None, *args) -> None:
    """
    PURPOSE:
    To format and print scraped data as markdown table 
    Special behaviour is fmt(false) which prints headers to the table
    
    ARGUMENTS:
    :param info: A dictionary containing the scraped data
    :param args: An array of strings that query the scraped data
    
    RETURNS: 
    :return: None
    
    EXAMPLE: 
    >>> fmt(False)
    |        KEY |   N_R |  CAT |      DST |     TIME | NAME  | URL
    """
    # Columns to format
    spacing = [
        ('KEY', 10, str.rjust),
        ('N_R', 5, str.rjust),
        ('CAT', 4, str.rjust),
        ('DST', 8, str.rjust),
        ('TIME', 8, str.rjust),
        ('NAME', 50, str.ljust),
        ('URL', 55, str.ljust),
        ('STATUS', 3, str.ljust),
    ]

    # Add empty formatters when too many args are given
    spacing += [(0, lambda x, y: x)] * (len(args) - len(spacing))

    if info is False:  # Print the spacers names as headers
        args = [k for k, _, _ in spacing]
    elif info is not None:  # Search the arg value in the info if possible
        args = [a if a not in info else info[a] for a in args]
        args = [f'{np.array(a)[np.array(a) > 0].mean():.1f} hrs' if isinstance(a, list) else a for a in args]

    # Apply the formatter to the args
    args = [str(a)[:sp] for a, (_, sp, _) in zip(args, spacing)]

    # Print the line, separated by pipes
    print('|', ' | '.join([mt(a, sp) for a, (_, sp, mt) in zip(args, spacing)]), '|')


def get_meta_info(request_response: requests.Response) -> dict:
    """   
    PURPOSE
    Extract metadata from the HTML content of a race event web page. 
    Identify and retrieve specific pieces of information such as the
    race title, category, and various other statistical details.

    ARGUMENTS
    request_response (requests.Response): The HTTP response object 

    RETURNS
    dict: Extracted metadata from the race event page. 
    
    EXAMPLE
    >>> print(get_meta_info(requests.get(https://utmb.world/utmb-index/races/12345..2024)))
    """
    # Some class names we will be looking for:
    title_classname = "font-24 font-d-34 futura-bold race-header_rh_race_title__COtYd"
    category_classname = "race-header_rh_category_logo_container__wTAh5"
    meta_classname = "col-12 col-md-6 col-lg-4 race-header_rh_stat_wrapper__1aSTO"

    # Get the page
    soup = BeautifulSoup(request_response.content, "html.parser")

    # Get the race title
    race_title = soup.find("h1", class_=title_classname).text

    # Get the race category
    race_category = '-'
    race_category_container = soup.find("div", class_=category_classname)
    if race_category_container is not None:
        race_category = race_category_container.find('img')["alt"]

    # Get race meta info
    meta_info_fields = soup.find_all("div", class_=meta_classname)
    meta_info = {inf.find("p").text: inf.find_all("span")[-1].text for inf in meta_info_fields}
    meta_info['Race Category'] = race_category
    meta_info['Race Title'] = race_title
    return meta_info


def add_page(request_url: str | bytes, info: dict) -> int:
    """
    PURPOSE
    The add_page function fetches and parses a web page containing race event results, 
    extracts relevant information, and updates a provided dictionary with this information. 
    It handles different scenarios such as no results, missing pages, and paginated data.

    ARGUMENTS
    request_url (str | bytes): The URL of the web page to be requested and parsed. This URL points to a specific race event page.
    info (dict): A dictionary to be updated with the extracted metadata and results. This dictionary will be populated with information such as race title, category, results, number of results, and breakdowns by country, sex, and age.
    
    RETURNS
    int: A status code indicating the outcome of the function:
    200: Successfully fetched and processed the page.
    201: No more results are available on the page.
    204: The page exists but contains no results.
    404: The requested page does not exist.
    
    EXAMPLE:
    # URL of the race event page
    >>> meta_info = {}
    >>> status = add_page("https://utmb.world/utmb-index/races/12345..2024?page=1", meta_info)
    Page processed successfully.
    >>> print(meta_info)
    {
        'Race Title': 'Example Race Title',
        ...
        'Age': {'Age Group1': count1, 'Age Group2': count2, ...}
    }
    """
    
    # Where to find page components
    row_classname = "my-table_cell__z__zN"
    n_results_classname = "font-16 font-d-18 font-oxanium-bold display-list-result_hit_qty__DPf3k"

    # Request the page
    response = requests.get(request_url)

    # No page exists for this year
    if response.status_code == 404:
        return 404

    soup = BeautifulSoup(response.content, "html.parser")
    if not any(info):
        n_results_str = soup.find_all("h2", class_=n_results_classname)[0].text

        # No results available
        if n_results_str == 'No results':
            return 204

        info.update(get_meta_info(response))
        info['Results'] = []
        info['N Results'] = int(n_results_str.split(' ')[0])
        info['Country'] = {}
        info['Sex'] = {}
        info['Age'] = {}

    # Get all results from this page
    rows = soup.find_all("div", class_="my-table_row__nlm_j")

    # We have exhausted the search for results for this race
    if not any(rows):
        return 201

    # Extract the information of each row
    for row in rows:
        rank, time, _, country, sex, age, _ = [i.text for i in row.find_all("div", class_=row_classname)]
        result = 0 if rank == 'DNF' else np.round(time2float(time), 4)
        info['Results'].append(result)
        for field, value in (('Country', country), ('Sex', sex), ('Age', age)):
            if value not in info[field]:
                info[field][value] = 0
            info[field][value] += 1
    # Continue to the next page
    return 200


# Define file paths
root = Path.cwd()
race_uids = list(map(int, json_in(root / 'utmb-race-uids.json')))
results_file = root / 'utmb-race-data-raw.json'

# Load previous results
utmb_results = json_in(results_file)
last_race_uid = np.max([int(i.split('.')[0]) for i in utmb_results])

# Print the header of our report
fmt(False)

for race_uid, year in product(race_uids, range(2014, 2025)):
    utmb_key = f'{race_uid}.{year}'

    # Check if this race_uid has already been done
    if race_uid < last_race_uid or utmb_key in utmb_results:
        continue

    # These values will be set during the while loop
    meta_info, status, page_no = {}, 200, 1
    while status == 200:
        # Request the web page
        url = f"https://utmb.world/utmb-index/races/{race_uid}..{year}?page={page_no}"
        status = add_page(url, meta_info)
        page_no += 1

    # Report and assign our product
    content = [''] * 5
    if meta_info is not None:
        utmb_results[utmb_key] = meta_info
        content = 'N Results', 'Race Category', 'Distance', 'Results', 'Race Title'
    fmt(meta_info, utmb_key, *content, url, status)

    # Save every 20 iterations
    if not len(utmb_results) % 20:
        json_out(utmb_results, results_file)
